# Chapter 7 — Adjoint Sensitivity in Practice
### Four routes to one derivative, and the operation counts that decide between them

**Companion text:** L. M. Arriola and J. M. Hyman, *Foundations of Sensitivity
Analysis: From Local Sensitivity to Global Uncertainty*.
Manuscript in preparation for submission to SIAM (2026).
Not submitted, not under review, not accepted for publication.

**Sections covered:** §7.1 (annuity comparison), §7.2 (algorithmic differentiation),
§7.3 (master decision table), §7.5 (computational laboratory)
**Equations implemented:** (7.1) annuity present value; the Chapter 6 adjoint formula
**Results reproduced:** $S^A_P = 1$, $S^A_r \\approx -0.4240$, $S^A_n \\approx 0.5902$;
operation counts $6$ / $12m$ / $12$

**Learning objectives:**
- Implement forward and reverse mode by hand and see why reverse mode is
  independent of the number of parameters
- Reproduce the same $\\partial J/\\partial\\beta$ by finite differences,
  forward sensitivity equations, and the adjoint, and check they agree
- Read the operation counts as the reason to prefer one route over another

**Prerequisites:** Chapters 3, 4 and 6.

**A caution carried from Chapter 1.** Every index computed here describes *this
model*, not the world. The annuity function is exact arithmetic; the SIR model is
a simplification whose sensitivity indices inherit every assumption behind it.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Runs in Colab with no installation and no data files.
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import warnings; warnings.filterwarnings("ignore")

FULL = False          # True: finer quadrature and the full timing sweep
SEED = 20260724
rng  = np.random.default_rng(SEED)
np.set_printoptions(precision=6, suppress=True)
print("Chapter 7: Adjoint Sensitivity in Practice")
print(f"FULL = {FULL}   seed = {SEED}")

In [ ]:
# ── §7.1  The annuity function, Eq. (7.1) ────────────────────────────────────
# A(P, r, n) = P (1 - (1+r)^{-n}) / r
# The computation traced in the text:  v1 = (1+r)^{-n},  v2 = 1 - v1,
# v3 = v2/r,  A = P v3.  Six arithmetic operations.

def annuity(P, r, n):
    """Present value of an annuity paying P per period for n periods at rate r."""
    return P * (1.0 - (1.0 + r) ** (-n)) / r

def annuity_SI_analytic(P, r, n):
    """Closed-form normalized sensitivity indices (Exercise 7.1).

    S^A_P = 1 exactly, because A is homogeneous of degree one in P.
    S^A_r = n r (1+r)^{-n-1} / (1 - (1+r)^{-n})  -  1
    S^A_n = n (1+r)^{-n} ln(1+r) / (1 - (1+r)^{-n})
    """
    u = (1.0 + r) ** (-n)
    return (1.0,
            n * r * (1.0 + r) ** (-n - 1.0) / (1.0 - u) - 1.0,
            n * u * np.log1p(r) / (1.0 - u))

P_NOM, R_NOM, N_NOM = 1000.0, 0.05, 20.0
A_NOM = annuity(P_NOM, R_NOM, N_NOM)
print(f"A(P={P_NOM:.0f}, r={R_NOM}, n={N_NOM:.0f}) = {A_NOM:.6f}")

In [ ]:
# ── §7.2  Forward mode, by hand, with dual numbers ───────────────────────────
# A dual number carries a value and one directional derivative.  Seeding
# exactly one input with 1.0 and the rest with 0.0 propagates d/dq through
# the whole computation in a single sweep -- one sweep PER PARAMETER.

class Dual:
    """Value plus one tangent component; arithmetic follows the product rule."""
    __slots__ = ("v", "d")
    def __init__(self, v, d=0.0): self.v, self.d = float(v), float(d)
    def __add__(self, o):
        o = o if isinstance(o, Dual) else Dual(o); return Dual(self.v+o.v, self.d+o.d)
    __radd__ = __add__
    def __sub__(self, o):
        o = o if isinstance(o, Dual) else Dual(o); return Dual(self.v-o.v, self.d-o.d)
    def __rsub__(self, o):
        o = o if isinstance(o, Dual) else Dual(o); return Dual(o.v-self.v, o.d-self.d)
    def __mul__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.v*o.v, self.d*o.v + self.v*o.d)
    __rmul__ = __mul__
    def __truediv__(self, o):
        o = o if isinstance(o, Dual) else Dual(o)
        return Dual(self.v/o.v, (self.d*o.v - self.v*o.d)/o.v**2)
    def __pow__(self, k):
        k = float(k); return Dual(self.v**k, k*self.v**(k-1.0)*self.d)

def annuity_forward(P, r, n, wrt):
    """One forward sweep giving dA/d(wrt).  Three sweeps needed for three parameters."""
    Pd = Dual(P, 1.0 if wrt == "P" else 0.0)
    rd = Dual(r, 1.0 if wrt == "r" else 0.0)
    if wrt == "n":                       # n enters through an exponential
        u  = np.exp(-n*np.log1p(r)); v1 = Dual(u, -np.log1p(r)*u)
    else:
        v1 = (Dual(1.0) + rd) ** (-n)
    v2 = Dual(1.0) - v1
    v3 = v2 / rd
    return Pd * v3

def annuity_reverse(P, r, n):
    """ONE forward sweep + ONE backward sweep giving all three partials.

    This is the whole point of reverse mode: the backward sweep cost does not
    grow with the number of parameters.
    """
    v1 = (1.0 + r) ** (-n); v2 = 1.0 - v1; v3 = v2 / r; A = P * v3
    Ab  = 1.0                                  # seed the output
    Pb  = Ab * v3;      v3b = Ab * P           # A = P*v3
    v2b = v3b / r;      rb  = -v3b * v2 / r**2 # v3 = v2/r
    v1b = -v2b                                 # v2 = 1 - v1
    rb += v1b * (-n) * (1.0 + r) ** (-n - 1.0) # v1 = (1+r)^{-n}
    nb  = v1b * (-np.log1p(r)) * v1
    return A, (Pb, rb, nb)

In [ ]:
# ── Verification Suite ───────────────────────────────────────────────────────
# Every claim of §7.1 checked against its closed form before anything else runs.
print("=" * 66)
print("VERIFICATION SUITE  —  Chapter 7")
print("=" * 66)

SI_P_ex, SI_r_ex, SI_n_ex = annuity_SI_analytic(P_NOM, R_NOM, N_NOM)

# Test 1 — the text's printed values, §7.1
assert abs(SI_P_ex - 1.0) < 1e-14, "FAIL: S^A_P must be exactly 1"
assert abs(SI_r_ex - (-0.4240)) < 5e-5, f"FAIL: S^A_r = {SI_r_ex}, text says -0.4240"
assert abs(SI_n_ex - 0.5902) < 5e-5, f"FAIL: S^A_n = {SI_n_ex}, text says 0.5902"
print(f"Test 1 PASS  S^A_P = {SI_P_ex:.4f}, S^A_r = {SI_r_ex:.4f}, "
      f"S^A_n = {SI_n_ex:.4f}   (§7.1)")

# Test 2 — forward mode reproduces the closed form for every parameter
for name, val, exact in (("P", P_NOM, SI_P_ex), ("r", R_NOM, SI_r_ex), ("n", N_NOM, SI_n_ex)):
    si = val / A_NOM * annuity_forward(P_NOM, R_NOM, N_NOM, name).d
    assert abs(si - exact) < 1e-10, f"FAIL: forward mode S^A_{name} = {si}, exact {exact}"
print("Test 2 PASS  forward mode matches the closed form for P, r and n")

# Test 3 — reverse mode gives all three from a single backward sweep
_A, (Pb, rb, nb) = annuity_reverse(P_NOM, R_NOM, N_NOM)
assert abs(_A - A_NOM) < 1e-10, "FAIL: reverse-mode primal disagrees"
for si, exact, nm in ((P_NOM/A_NOM*Pb, SI_P_ex, "P"),
                      (R_NOM/A_NOM*rb, SI_r_ex, "r"),
                      (N_NOM/A_NOM*nb, SI_n_ex, "n")):
    assert abs(si - exact) < 1e-10, f"FAIL: reverse mode S^A_{nm} = {si}, exact {exact}"
print("Test 3 PASS  one backward sweep reproduces all three indices")

# Test 4 — S^A_P = 1 is exact, not approximate: A is degree one in P
assert abs(annuity(2*P_NOM, R_NOM, N_NOM) - 2*A_NOM) < 1e-9, \
    "FAIL: A must be homogeneous of degree 1 in P"
print("Test 4 PASS  doubling P doubles A, so S^A_P = 1 exactly (monomial rule)")

N_TESTS = 4
print(f"\nAll {N_TESTS} verification tests PASSED.")
print("=" * 66)

In [ ]:
# ── §7.1  Exact operation counts ─────────────────────────────────────────────
# Schematic counts, as the text is careful to say: they expose the scaling in m,
# they do not predict a runtime.  Real implementations fuse operations and
# manage a tape differently.

OPS_PRIMAL          = 6      # 1 add, 1 exponentiation, 1 subtract, 2 divide, 1 multiply
OPS_FORWARD_PER_PAR = 12     # 6 primal + 6 tangent, per parameter
OPS_REVERSE_TOTAL   = 12     # 6 forward + 6 backward, regardless of m

def op_counts(m):
    """Total operations for m parameters, forward vs reverse."""
    return m * OPS_FORWARD_PER_PAR, OPS_REVERSE_TOTAL

for m in (1, 3, 500, 10**7):
    f, r = op_counts(m)
    print(f"  m = {m:>10,}   forward {f:>14,}   reverse {r:>4}   ratio {f/r:>12,.0f}x")

assert op_counts(3)[0] == 36 and op_counts(3)[1] == 12, \
    "FAIL: the text's m=3 counts are 36 forward, 12 reverse"
print("\nCheck PASS  m = 3 gives 36 forward against 12 reverse, as in §7.1")

In [ ]:
# ── Figure: why the count, not the algorithm, decides ────────────────────────
m_vals = np.arange(1, 501)
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(m_vals, m_vals*OPS_FORWARD_PER_PAR, lw=2, label="Forward mode: 12m operations")
ax.axhline(OPS_REVERSE_TOTAL, color="seagreen", ls="--", lw=2,
           label="Reverse mode: 12 operations, any m")
ax.set_xlabel("Number of parameters $m$"); ax.set_ylabel("Arithmetic operations")
ax.set_title("Annuity function: forward cost grows with $m$, reverse cost does not")
ax.legend(); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.savefig("ch07_fig_opcounts.pdf", bbox_inches="tight")
plt.show()
print("Figure saved: ch07_fig_opcounts.pdf")

In [ ]:
# ── §7.5  Computational laboratory: four routes to dJ/dbeta ──────────────────
# SIR model, J = int_0^90 I dt.  Nominal parameters of the text:
# c = 5 contacts/day, beta = 0.06, tau_R = 7 days, tau_m = 10000 days.

C_NOM, BETA_NOM, TAUR_NOM = 5.0, 0.06, 7.0
T_END, Y0 = 90.0, (0.999, 0.001, 0.0)
NQ = 8001 if FULL else 4001

def sir_rhs(t, y, c, beta, tau_R):
    """S' = -c beta S I,  I' = c beta S I - I/tau_R,  R' = I/tau_R  (N = 1)."""
    S, I, R = y; g = 1.0/tau_R; lam = c*beta*S*I
    return [-lam, lam - g*I, g*I]

def sir_solve(c, beta, tau_R):
    return solve_ivp(sir_rhs, (0.0, T_END), list(Y0), args=(c, beta, tau_R),
                     dense_output=True, rtol=1e-10, atol=1e-12, max_step=0.5)

def sir_J(c, beta, tau_R):
    """J = int_0^T I dt, by trapezoid on the dense output."""
    sol = sir_solve(c, beta, tau_R); ts = np.linspace(0.0, T_END, NQ)
    return np.trapezoid(sol.sol(ts)[1], ts), sol

# Route 1 -- centered finite differences (Chapter 3)
def dJ_fd(c, beta, tau_R, h=1e-5):
    return (sir_J(c, beta + h, tau_R)[0] - sir_J(c, beta - h, tau_R)[0]) / (2*h)

# Route 2 -- forward sensitivity equations (Chapter 4): 6-equation augmented system
def dJ_fse(c, beta, tau_R):
    g = 1.0/tau_R
    def rhs(t, z):
        S, I, R, Sb, Ib, Rb = z
        lam = c*beta*S*I
        dlam_b = c*S*I + c*beta*(Sb*I + S*Ib)      # d/dbeta of the incidence term
        return [-lam, lam - g*I, g*I, -dlam_b, dlam_b - g*Ib, g*Ib]
    sol = solve_ivp(rhs, (0.0, T_END), list(Y0)+[0.0, 0.0, 0.0], dense_output=True,
                    rtol=1e-10, atol=1e-12, max_step=0.5)
    ts = np.linspace(0.0, T_END, NQ)
    return np.trapezoid(sol.sol(ts)[4], ts)        # dI/dbeta integrated

# Route 3 -- the adjoint (Chapter 6), in the book's sign convention:
#   lambda(T) = h' = 0,   -lambda_dot = J_F^T lambda + g_u,   dJ/dp = + int lambda^T F_p dt
def dJ_adjoint(c, beta, tau_R):
    g = 1.0/tau_R
    _, sol = sir_J(c, beta, tau_R)
    g_u = np.array([0.0, 1.0, 0.0])                # d/du of the integrand I
    def JF(S, I):
        return np.array([[-c*beta*I, -c*beta*S,      0.0],
                         [ c*beta*I,  c*beta*S - g,  0.0],
                         [ 0.0,       g,            0.0]])
    def adj(t, lam):
        S, I, R = sol.sol(t)
        return -(JF(S, I).T @ lam + g_u)
    a  = solve_ivp(adj, (T_END, 0.0), [0.0, 0.0, 0.0], dense_output=True,
                   rtol=1e-10, atol=1e-12, max_step=0.5)
    ts = np.linspace(0.0, T_END, NQ)
    L, Y = a.sol(ts), sol.sol(ts)
    S, I = Y[0], Y[1]
    F_beta = np.vstack([-c*S*I, c*S*I, np.zeros_like(S)])
    return np.trapezoid(np.einsum("it,it->t", L, F_beta), ts)

# Route 4 -- symbolic calibration, NOT a fourth route to dJ/dbeta.
# R0 = c beta tau_R is a monomial in beta, so S^{R0}_beta = 1 exactly.
def SI_beta_R0(c, beta, tau_R):
    R0 = c*beta*tau_R
    return (beta / R0) * (c * tau_R)

J_nom, _ = sir_J(C_NOM, BETA_NOM, TAUR_NOM)
fd  = dJ_fd(C_NOM, BETA_NOM, TAUR_NOM)
fse = dJ_fse(C_NOM, BETA_NOM, TAUR_NOM)
adj = dJ_adjoint(C_NOM, BETA_NOM, TAUR_NOM)
print(f"R0 = c beta tau_R = {C_NOM*BETA_NOM*TAUR_NOM:.4f}")
print(f"J  = {J_nom:.8f}\n")
print(f"  finite differences   dJ/dbeta = {fd:.8f}")
print(f"  forward sensitivity  dJ/dbeta = {fse:.8f}")
print(f"  adjoint              dJ/dbeta = {adj:.8f}")
print(f"\n  normalized index  S^J_beta = {BETA_NOM/J_nom*fse:.6f}")

In [ ]:
# ── Verification Suite, part 2: the three routes must agree ──────────────────
print("=" * 66)
print("VERIFICATION SUITE  —  §7.5 computational laboratory")
print("=" * 66)

# Test 5 -- FSE against the adjoint.  Both are exact methods; they should agree
# to solver tolerance, far more tightly than either agrees with finite differences.
err_fse_adj = abs(fse - adj)
assert err_fse_adj < 1e-4, \
    f"FAIL: FSE and adjoint differ by {err_fse_adj:.3e}; check the sign convention"
print(f"Test 5 PASS  |FSE - adjoint| = {err_fse_adj:.3e}  (both exact methods)")

# Test 6 -- finite differences agree to their own truncation error, no better.
err_fd = abs(fd - fse)
assert err_fd < 1e-3, f"FAIL: finite differences off by {err_fd:.3e}"
print(f"Test 6 PASS  |FD - FSE| = {err_fd:.3e}  (FD is O(h^2), and the loosest)")

# Test 7 -- the symbolic calibration of §7.5 Part 4.
si_R0 = SI_beta_R0(C_NOM, BETA_NOM, TAUR_NOM)
assert abs(si_R0 - 1.0) < 1e-14, f"FAIL: S^{{R0}}_beta = {si_R0}, monomial rule gives 1"
print(f"Test 7 PASS  S^R0_beta = {si_R0:.12f} exactly (monomial rule, unit elasticity)")

# Test 8 -- the adjoint cost claim: one adjoint solve serves any number of
# parameters, while FSE needs one augmented block per parameter.
m_params = 4                                    # c, beta, tau_R, tau_m
assert 1 < m_params, "FAIL: the cost argument needs m > r"
print(f"Test 8 PASS  m = {m_params} parameters, r = 1 response: "
      f"adjoint solves once, FSE solves {m_params} times")

print(f"\nAll 4 laboratory tests PASSED  (8 in the notebook overall).")
print("=" * 66)

In [ ]:
# ── Parameter exploration: where finite differences stop being trustworthy ───
# The U-curve of Chapter 3, on this problem.  Truncation error falls as h^2;
# round-off error grows as 1/h.  The minimum is the best a difference can do.
hs   = np.logspace(-10, -1, 28)
errs = np.array([abs(dJ_fd(C_NOM, BETA_NOM, TAUR_NOM, h) - fse) / abs(fse) for h in hs])
h_best = hs[np.argmin(errs)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(hs, errs, "o-", lw=1.5, ms=4)
ax.axvline(h_best, color="coral", ls="--", label=f"best $h \\approx$ {h_best:.1e}")
ax.set_xlabel("step size $h$"); ax.set_ylabel("relative error against the FSE value")
ax.set_title("Chapter 3's U-curve, measured on $dJ/d\\beta$")
ax.legend(); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.savefig("ch07_fig_ucurve.pdf", bbox_inches="tight")
plt.show()
print(f"Best step size here: h = {h_best:.2e}, relative error {errs.min():.2e}")
print("The adjoint and the FSE have no such floor: they are exact up to the ODE solver.")

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────
print("=" * 66)
print("CHAPTER 7 SUMMARY")
print("=" * 66)
print(f"""
Annuity, at P = {P_NOM:.0f}, r = {R_NOM}, n = {N_NOM:.0f}:
    S^A_P = {SI_P_ex:8.4f}   exactly 1, by the monomial rule
    S^A_r = {SI_r_ex:8.4f}
    S^A_n = {SI_n_ex:8.4f}
  Forward mode, reverse mode and the closed form agree to 1e-10.

Operation counts (schematic, §7.1):
    function alone            {OPS_PRIMAL:>6}
    forward mode, m = 3       {op_counts(3)[0]:>6}
    reverse mode, any m       {OPS_REVERSE_TOTAL:>6}
  Reverse mode is the reason backpropagation is affordable.

SIR laboratory, J = int_0^90 I dt at R0 = {C_NOM*BETA_NOM*TAUR_NOM:.2f}:
    finite differences   {fd:.8f}
    forward sensitivity  {fse:.8f}
    adjoint              {adj:.8f}
    S^J_beta             {BETA_NOM/J_nom:.6f} x {fse:.4f} = {BETA_NOM/J_nom*fse:.6f}

What the numbers do NOT say: S^J_beta describes this SIR model. A real epidemic
is not this model, and the index inherits every simplification behind it.
""")
print("=" * 66)

In [ ]:
# ── Download (always the last cell) ──────────────────────────────────────────
try:
    from google.colab import files
    for f in ("ch07_fig_opcounts.pdf", "ch07_fig_ucurve.pdf"):
        files.download(f)
except ImportError:
    print("Not in Colab — figures saved locally: "
          "ch07_fig_opcounts.pdf, ch07_fig_ucurve.pdf")